# Regularization Benchmark: L2 vs Dropout

## Overview

In [deep-neural_network_cnn](deep-neural_network_cnn.ipynb), our CNN reached strong test accuracy, but it also memorized the training set almost perfectly. That train/test gap is the classic sign that the model has enough capacity to overfit.

This notebook keeps the **same CNN backbone**, the **same MNIST subset** and the **same training loop**, then compares four variants:

| Variant | L2 (weight decay) | Dropout |
|---------|-------------------|---------|
| CNN baseline | No | No |
| CNN + L2 | Yes | No |
| CNN + Dropout | No | Yes |
| CNN + L2 + Dropout | Yes | Yes |

### Our Journey So Far

| File | Topic | Key Concept |
|------|-------|-------------|
| [first_neural_network](first_neural_network.ipynb) | Neural Networks | Backpropagation |
| [deep-neural_network_numpy](deep-neural_network_numpy.ipynb) | Deep Networks (NumPy) | ReLU, softmax |
| [deep-neural_network_pytorch](deep-neural_network_pytorch.ipynb) | Fully connected (PyTorch) | Autograd, DataLoader |
| [deep-neural_network_cnn](deep-neural_network_cnn.ipynb) | CNN | Convolution, pooling |
| **deep-neural_network_regularization** | **Regularization** | **L2, Dropout, generalization** |

### What You'll Learn

- What L2 regularization (weight decay) does to the learning dynamics
- How Dropout reduces co-adaptation between neurons
- How to measure generalization with the train/test gap
- Whether combining both regularizers helps more than using one alone

### Hypothesis

- **L2** should encourage smaller weights and a smoother decision function, which often improves test performance when the model overfits.
- **Dropout** should force the network to distribute evidence across many neurons, which can also improve robustness on unseen digits.


## 1. Setup and Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12


## 2. Loading the MNIST Dataset

Same protocol as [deep-neural_network_cnn](deep-neural_network_cnn.ipynb): 10,000 samples, 80/20 split, seed 42. Keeping the data fixed is essential for a fair regularization benchmark.


In [ ]:
print("Loading MNIST dataset (this may take a moment)...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

X_full = mnist.data / 255.0
y_full = mnist.target.astype(int)

print("\nDataset loaded!")
print(f"X shape: {X_full.shape}")
print(f"Classes: {np.unique(y_full)}")


In [ ]:
n_samples = 10000
X_subset = X_full[:n_samples]
y_subset = y_full[:n_samples]

X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")


In [ ]:
BATCH_SIZE = 64

X_train_img = torch.FloatTensor(X_train.reshape(-1, 1, 28, 28))
X_test_img = torch.FloatTensor(X_test.reshape(-1, 1, 28, 28))
y_train_t = torch.LongTensor(y_train)
y_test_t = torch.LongTensor(y_test)

train_loader = DataLoader(
    TensorDataset(X_train_img, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(X_test_img, y_test_t),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Batch X shape: {next(iter(train_loader))[0].shape}")
print(f"Training batches: {len(train_loader)}")


## 3. What Are We Trying to Fix?

A model that reaches ~100% train accuracy and a lower test accuracy has learned patterns that do not fully transfer. Two common responses:

### L2 regularization (weight decay)

We add a penalty on the squared magnitude of the weights:

$$J_{\text{reg}} = J_{\text{data}} + \frac{\lambda}{2}\sum_i w_i^2$$

In practice with SGD, this is implemented as `weight_decay` in the optimizer. Large weights are gently pulled toward zero, which tends to produce a smoother predictor and reduce overfitting.

### Dropout

During training, each selected neuron is randomly zeroed with probability `p`. The network cannot rely on a single lucky unit and must learn redundant, more robust representations. At evaluation time, dropout is disabled and activations are scaled automatically by PyTorch.


In [ ]:
# Intuition sketch: L2 prefers smaller weights
weights = np.linspace(-3, 3, 400)
l2_penalty = 0.5 * weights ** 2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(weights, l2_penalty, color='#e74c3c', linewidth=2)
axes[0].set_title('L2 penalty grows with |weight|')
axes[0].set_xlabel('Weight value')
axes[0].set_ylabel('Penalty')

# Dropout intuition: random masks force distributed representations
rng = np.random.default_rng(42)
activations = np.ones(12)
mask = rng.random(12) > 0.5
axes[1].bar(range(12), activations, color='#95a5a6', label='Active before dropout')
axes[1].bar(range(12), activations * mask, color='#3498db', label='Kept after dropout')
axes[1].set_title('Dropout randomly drops neurons (p=0.5)')
axes[1].set_xlabel('Neuron index')
axes[1].set_ylabel('Activation kept')
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Shared CNN Backbone

All four variants share the same convolutional architecture as the previous notebook:

`(1, 28, 28) → Conv16 → Pool → Conv32 → Pool → FC128 → 10`

The only differences are:

- whether `nn.Dropout` is inserted after the dense ReLU
- whether SGD uses `weight_decay` (L2)


In [ ]:
class RegularizedCNN(nn.Module):
    """
    Same CNN backbone for all benchmark variants.
    Toggle dropout with dropout_p > 0.
    """

    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        if x.ndim == 2:
            x = x.view(-1, 1, 28, 28)
        x = self.features(x)
        return self.classifier(x)


# Inspect one model with dropout enabled
demo = RegularizedCNN(dropout_p=0.5)
print(demo)
print(f"\nTrainable parameters: {sum(p.numel() for p in demo.parameters()):,}")


## 5. Benchmark Configuration

We compare four controlled settings. Everything else stays identical: data, batch size, learning rate, epochs and random seed before each run.


In [ ]:
EPOCHS = 30
LR = 0.1
WEIGHT_DECAY = 1e-4   # L2 strength
DROPOUT_P = 0.5       # dropout probability on the FC hidden layer

configs = [
    {"name": "CNN", "dropout_p": 0.0, "weight_decay": 0.0, "color": "#3498db"},
    {"name": "CNN + L2", "dropout_p": 0.0, "weight_decay": WEIGHT_DECAY, "color": "#2ecc71"},
    {"name": "CNN + Dropout", "dropout_p": DROPOUT_P, "weight_decay": 0.0, "color": "#e67e22"},
    {"name": "CNN + L2 + Dropout", "dropout_p": DROPOUT_P, "weight_decay": WEIGHT_DECAY, "color": "#e74c3c"},
]

print("Benchmark configurations:")
print("-" * 70)
print(f"{'Variant':<22} {'Dropout p':<12} {'Weight decay (L2)':<20}")
print("-" * 70)
for cfg in configs:
    print(f"{cfg['name']:<22} {cfg['dropout_p']:<12} {cfg['weight_decay']:<20}")
print("-" * 70)
print(f"Shared settings: lr={LR}, epochs={EPOCHS}, batch_size={BATCH_SIZE}")
print("Note: 30 epochs keep the 4-way benchmark practical on CPU while remaining comparable.")


## 6. Training Utilities

The training loop is the same for every variant. Regularization enters in only two places:

1. model construction (`Dropout`)
2. optimizer construction (`weight_decay`)


In [ ]:
def evaluate(model, data_loader, max_batches=None):
    """Evaluate loss/accuracy. Optionally limit batches for faster train monitoring."""
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(data_loader):
            if max_batches is not None and i >= max_batches:
                break
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss_sum += criterion(logits, y_batch).item() * y_batch.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return loss_sum / total, correct / total


def train_variant(name, dropout_p, weight_decay, lr=LR, epochs=EPOCHS, print_every=5):
    torch.manual_seed(42)
    model = RegularizedCNN(dropout_p=dropout_p).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'gap': [],
    }

    print(f"\nTraining {name}...")
    print(f"  dropout_p={dropout_p}, weight_decay={weight_decay}, lr={lr}, epochs={epochs}")
    print("=" * 70)

    for epoch in range(epochs):
        model.train()  # enables dropout during training
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

        # Full test eval; lighter train probe (~20 batches) to keep the benchmark fast
        train_loss, train_acc = evaluate(model, train_loader, max_batches=20)
        test_loss, test_acc = evaluate(model, test_loader)
        gap = train_acc - test_acc

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['gap'].append(gap)

        if epoch % print_every == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch + 1:3d}/{epochs}: "
                f"Train {train_acc:.2%} | Test {test_acc:.2%} | Gap {gap:+.2%}"
            )

    # Final metrics on the full train set for a fair scoreboard
    final_train_loss, final_train_acc = evaluate(model, train_loader)
    final_test_loss, final_test_acc = evaluate(model, test_loader)
    history['train_loss'][-1] = final_train_loss
    history['train_acc'][-1] = final_train_acc
    history['test_loss'][-1] = final_test_loss
    history['test_acc'][-1] = final_test_acc
    history['gap'][-1] = final_train_acc - final_test_acc

    print("=" * 70)
    print(
        f"Final ({name}): Train {history['train_acc'][-1]:.2%} | "
        f"Test {history['test_acc'][-1]:.2%} | Gap {history['gap'][-1]:+.2%}"
    )
    return model, history


## 7. Run the Benchmark


In [ ]:
results = {}
models = {}

for cfg in configs:
    model, history = train_variant(
        name=cfg['name'],
        dropout_p=cfg['dropout_p'],
        weight_decay=cfg['weight_decay'],
    )
    results[cfg['name']] = history
    models[cfg['name']] = model


## 8. Learning Curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for cfg in configs:
    name = cfg['name']
    color = cfg['color']
    hist = results[name]

    axes[0].plot(hist['train_acc'], color=color, linewidth=2, label=f'{name} train')
    axes[0].plot(hist['test_acc'], color=color, linewidth=2, linestyle='--', label=f'{name} test')

    axes[1].plot(hist['test_acc'], color=color, linewidth=2, label=name)
    axes[2].plot(hist['gap'], color=color, linewidth=2, label=name)

axes[0].set_title('Train vs Test Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8)

axes[1].set_title('Test Accuracy Only')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test accuracy')
axes[1].legend()

axes[2].set_title('Generalization Gap (Train - Test)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Gap')
axes[2].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[2].legend()

plt.tight_layout()
plt.show()


## 9. Final Scoreboard


In [ ]:
print("=" * 88)
print("REGULARIZATION BENCHMARK — FINAL RESULTS")
print("=" * 88)
print(
    f"{'Variant':<22} {'Train Acc':<12} {'Test Acc':<12} "
    f"{'Gap':<12} {'Best Test':<12} {'Params':<10}"
)
print("-" * 88)

scoreboard = []
for cfg in configs:
    name = cfg['name']
    hist = results[name]
    train_acc = hist['train_acc'][-1]
    test_acc = hist['test_acc'][-1]
    gap = hist['gap'][-1]
    best_test = max(hist['test_acc'])
    n_params = sum(p.numel() for p in models[name].parameters())
    scoreboard.append(
        {
            'name': name,
            'train_acc': train_acc,
            'test_acc': test_acc,
            'gap': gap,
            'best_test': best_test,
            'params': n_params,
        }
    )
    print(
        f"{name:<22} {train_acc:<12.2%} {test_acc:<12.2%} "
        f"{gap:<+12.2%} {best_test:<12.2%} {n_params:<10,}"
    )

print("=" * 88)

baseline = scoreboard[0]
print("\nDeltas vs CNN baseline (final test accuracy):")
for row in scoreboard[1:]:
    delta = row['test_acc'] - baseline['test_acc']
    gap_delta = row['gap'] - baseline['gap']
    print(
        f"  {row['name']:<22} test {delta:+.2%} | "
        f"gap change {gap_delta:+.2%} "
        f"({'smaller gap' if gap_delta < 0 else 'larger gap'})"
    )


In [ ]:
names = [row['name'] for row in scoreboard]
test_accs = [row['test_acc'] for row in scoreboard]
gaps = [row['gap'] for row in scoreboard]
colors = [cfg['color'] for cfg in configs]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bars = axes[0].bar(names, test_accs, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_ylim(0.9, 1.0)
axes[0].set_ylabel('Final test accuracy')
axes[0].set_title('Test Accuracy by Variant')
axes[0].tick_params(axis='x', rotation=20)
for bar, acc in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f"{acc:.2%}", ha='center', va='bottom', fontsize=10)

bars = axes[1].bar(names, gaps, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Train - Test gap')
axes[1].set_title('Generalization Gap (lower is better)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].axhline(0, color='gray', linestyle=':')
for bar, gap in zip(bars, gaps):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                 f"{gap:.2%}", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


## 10. Weight Magnitudes: Does L2 Really Shrink Weights?


In [ ]:
def mean_abs_weight(model):
    vals = []
    for p in model.parameters():
        if p.ndim >= 2:  # skip biases
            vals.append(p.detach().abs().mean().item())
    return float(np.mean(vals))


print("Mean absolute weight (excluding biases):")
print("-" * 50)
for cfg in configs:
    name = cfg['name']
    maw = mean_abs_weight(models[name])
    print(f"  {name:<22} {maw:.5f}")
print("-" * 50)
print("Expectation: variants with L2 should show smaller average |weights|.")


## 11. Per-Class Test Accuracy


In [ ]:
def predict_all(model, data_loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds.append(logits.argmax(dim=1).cpu().numpy())
            trues.append(y_batch.numpy())
    return np.concatenate(preds), np.concatenate(trues)


true_y = None
per_class = {}
for cfg in configs:
    name = cfg['name']
    preds, true_y = predict_all(models[name], test_loader)
    conf = np.zeros((10, 10), dtype=int)
    for t, p in zip(true_y, preds):
        conf[t, p] += 1
    per_class[name] = [conf[i, i] / conf[i].sum() for i in range(10)]

print(f"{'Digit':<8}" + "".join(f"{cfg['name'][:16]:<18}" for cfg in configs))
print("-" * 80)
for digit in range(10):
    row = f"{digit:<8}"
    for cfg in configs:
        row += f"{per_class[cfg['name']][digit]:<18.1%}"
    print(row)


## 12. How to Read These Results

### If L2 helps

You should see a **smaller generalization gap** and often a **higher or more stable test accuracy**. The weight-magnitude check should also show smaller average `|w|`. That matches the idea of a softer learning dynamics: the model is discouraged from fitting brittle, high-magnitude solutions.

### If Dropout helps

Train accuracy usually rises more slowly, because the network is constantly missing random neurons. Test accuracy can still improve if the ensemble-like effect reduces co-adaptation. Dropout is most useful when the dense head over-relies on a few units.

### If L2 + Dropout wins

The two regularizers attack overfitting differently: L2 reshapes the weight landscape globally, Dropout reshapes the computational graph locally during training. Together they can compound, but they can also over-regularize on an easy dataset like MNIST.

### Important caveat for this project

MNIST digits are clean and centered. With an already strong CNN, absolute gains may be small. The most informative metric is often the **gap**, not only the raw test score.


## 13. Strengths and Limits of Each Regularizer

### L2 / weight decay

**Strengths**
- Simple to add (`weight_decay` in the optimizer)
- Encourages smoother decision functions
- Often reduces the train/test gap without changing architecture

**Limits**
- One global hyperparameter `λ` for the whole network
- Too strong an L2 can underfit and hurt test accuracy
- Does not explicitly break neuron co-adaptation

### Dropout

**Strengths**
- Directly reduces reliance on individual neurons
- Acts like a cheap ensemble during training
- Particularly useful in dense layers with many parameters

**Limits**
- Slows training convergence
- Needs `model.train()` / `model.eval()` handled correctly
- Too high a dropout rate can destroy useful signal

### Combined L2 + Dropout

Useful when overfitting is clear and each regularizer alone is insufficient. On small, clean datasets, the combination can become too aggressive and slightly degrade test performance.


## 14. Summary

### What We Benchmarked

| Variant | Mechanism | Question answered |
|---------|-----------|-------------------|
| CNN | No extra regularization | Baseline generalization |
| CNN + L2 | Shrink weights | Does a softer weight penalty help test accuracy? |
| CNN + Dropout | Randomly drop neurons | Does reduced co-adaptation help? |
| CNN + L2 + Dropout | Both | Do the two effects stack? |

### Practical Takeaway

Regularization is not about making training look better. It is about improving behavior on data the model has never seen. In this notebook, judge each variant by:

1. final test accuracy
2. train/test gap
3. stability of the test curve

### What's Next

- Sweep `weight_decay` and `dropout_p` more systematically
- Add data augmentation (another strong regularizer for images)
- Try BatchNorm as an alternative stabilizer
- Re-run the benchmark on the full 70,000 MNIST images


In [ ]:
print("=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)
print(f"Dataset:         MNIST subset (10,000 samples)")
print(f"Train / Test:    {len(y_train):,} / {len(y_test):,}")
print(f"Backbone:        Conv16 → Conv32 → FC128 → 10")
print(f"Epochs / LR:     {EPOCHS} / {LR}")
print(f"L2 strength:     {WEIGHT_DECAY}")
print(f"Dropout p:       {DROPOUT_P}")
print("-" * 70)
for row in scoreboard:
    print(
        f"{row['name']:<22} "
        f"test={row['test_acc']:.2%} | gap={row['gap']:+.2%}"
    )
best = max(scoreboard, key=lambda r: r['test_acc'])
smallest_gap = min(scoreboard, key=lambda r: r['gap'])
print("-" * 70)
print(f"Best test accuracy:   {best['name']} ({best['test_acc']:.2%})")
print(f"Smallest gap:         {smallest_gap['name']} ({smallest_gap['gap']:+.2%})")
print("=" * 70)
